# 📘 Declarative Automation Bundles (DABs)
## CI/CD, Deployment & Infrastructure-as-Code for Databricks

**What you'll learn:**
- What DABs are (formerly Databricks Asset Bundles)
- Bundle structure (databricks.yml, resources, src)
- Defining jobs, pipelines, and clusters as code
- Environment management (dev, staging, prod)
- CI/CD integration (GitHub Actions, Azure DevOps)
- Deployment workflows (validate → deploy → run)
- Variables, permissions, and overrides

**Prerequisites:** Notebooks 28 (CI/CD), 43 (Git)
**Study Order:** 19

---

---
# 📗 Section 1: What are DABs?

## The Problem DABs Solve

Without DABs, deploying Databricks resources means:
- Clicking through the UI to create jobs
- Manually configuring clusters for each environment
- No version control for job definitions
- No way to replicate environments consistently

With DABs:
- Define ALL resources as YAML code
- Version control everything in Git
- Deploy consistently across environments
- CI/CD automation (no manual clicks)

## Bundle Structure

```
my-pipeline/
├── databricks.yml              ← Main configuration
├── resources/
│   ├── jobs/
│   │   └── daily_pipeline.yml  ← Job definition
│   └── pipelines/
│       └── dlt_pipeline.yml    ← DLT pipeline definition
├── src/
│   ├── ingest.py               ← Pipeline code
│   ├── transform.py
│   └── quality_checks.py
├── tests/
│   └── test_transform.py
└── .github/
    └── workflows/
        └── deploy.yml          ← CI/CD workflow
```

## databricks.yml Example

```yaml
bundle:
  name: techmart-pipeline

workspace:
  host: https://my-workspace.cloud.databricks.com

resources:
  jobs:
    daily_revenue:
      name: "Daily Revenue Pipeline"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "America/New_York"
      tasks:
        - task_key: ingest
          notebook_task:
            notebook_path: ./src/ingest.py
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 4
        - task_key: transform
          depends_on:
            - task_key: ingest
          notebook_task:
            notebook_path: ./src/transform.py

targets:
  dev:
    workspace:
      host: https://dev-workspace.cloud.databricks.com
    resources:
      jobs:
        daily_revenue:
          name: "[DEV] Daily Revenue Pipeline"
  
  prod:
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    resources:
      jobs:
        daily_revenue:
          name: "[PROD] Daily Revenue Pipeline"
          tasks:
            - task_key: ingest
              new_cluster:
                num_workers: 16  # More workers in prod
```

## CLI Commands

```bash
# Validate bundle configuration
databricks bundle validate

# Deploy to target environment
databricks bundle deploy --target dev
databricks bundle deploy --target prod

# Run a job
databricks bundle run daily_revenue --target dev

# Destroy resources (cleanup)
databricks bundle destroy --target dev
```

In [0]:
# Simulate DABs deployment workflow
class DABsSimulator:
    """Simulates Declarative Automation Bundles deployment."""
    
    def __init__(self, bundle_name):
        self.bundle_name = bundle_name
        self.resources = {}
        self.targets = {}
        self.deployed = {}
    
    def define_job(self, name, tasks, schedule=None):
        self.resources[name] = {"type": "job", "tasks": tasks, "schedule": schedule}
    
    def define_target(self, name, overrides=None):
        self.targets[name] = overrides or {}
    
    def validate(self, target):
        """Validate bundle configuration."""
        errors = []
        if target not in self.targets:
            errors.append(f"Target '{target}' not defined")
        if not self.resources:
            errors.append("No resources defined")
        
        if errors:
            print(f"   ❌ Validation FAILED: {errors}")
            return False
        print(f"   ✅ Validation passed for target: {target}")
        return True
    
    def deploy(self, target):
        """Deploy resources to target environment."""
        if not self.validate(target):
            return False
        
        print(f"   🚀 Deploying to {target}...")
        for name, resource in self.resources.items():
            env_name = f"[{target.upper()}] {name}"
            self.deployed[f"{target}/{name}"] = {
                "name": env_name,
                "status": "deployed",
                "resource": resource
            }
            print(f"      Deployed: {env_name}")
        return True
    
    def status(self):
        """Show deployment status."""
        print(f"\n📋 Deployment Status:")
        for key, info in self.deployed.items():
            print(f"   {key}: {info['name']} ({info['status']})")


# Demo
dabs = DABsSimulator("techmart-pipeline")
dabs.define_job("daily_revenue", tasks=["ingest", "transform", "quality_check"], schedule="0 6 * * *")
dabs.define_job("hourly_cdc", tasks=["capture_changes", "apply_merge"], schedule="0 * * * *")
dabs.define_target("dev", {"workers": 2})
dabs.define_target("staging", {"workers": 4})
dabs.define_target("prod", {"workers": 16})

print("📦 DABs DEPLOYMENT WORKFLOW")
print("=" * 60)

# Deploy to dev
print("\n--- Deploy to DEV ---")
dabs.deploy("dev")

# Deploy to prod
print("\n--- Deploy to PROD ---")
dabs.deploy("prod")

dabs.status()


---
# 📗 Section 2: DABs Deep Dive

## What are Declarative Automation Bundles?

DABs (formerly Databricks Asset Bundles) let you define ALL Databricks resources
as code -- jobs, pipelines, clusters, permissions -- and deploy them consistently.

## Bundle Structure

```
my-pipeline/
├── databricks.yml          # Main configuration
├── resources/
│   ├── jobs/
│   │   └── daily_pipeline.yml
│   └── pipelines/
│       └── medallion.yml
├── src/
│   ├── ingest.py
│   ├── transform.py
│   └── quality_checks.py
├── tests/
│   └── test_transforms.py
└── .github/
    └── workflows/
        └── deploy.yml
```

## databricks.yml Reference

```yaml
bundle:
  name: techmart-pipeline

workspace:
  host: ${var.databricks_host}

variables:
  databricks_host:
    description: "Databricks workspace URL"
  env:
    description: "Deployment environment"
    default: dev

resources:
  jobs:
    daily_revenue:
      name: "[${var.env}] Daily Revenue Pipeline"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "America/New_York"
      email_notifications:
        on_failure: ["data-team@company.com"]
      tasks:
        - task_key: ingest
          notebook_task:
            notebook_path: ./src/ingest
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 4
        - task_key: transform
          depends_on: [{task_key: ingest}]
          notebook_task:
            notebook_path: ./src/transform

targets:
  dev:
    mode: development
    workspace:
      host: https://dev-workspace.cloud.databricks.com
  prod:
    mode: production
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    resources:
      jobs:
        daily_revenue:
          tasks:
            - task_key: ingest
              new_cluster:
                num_workers: 16  # More workers in prod
```

## CLI Commands

```bash
# Validate configuration
databricks bundle validate --target dev

# Deploy resources
databricks bundle deploy --target dev
databricks bundle deploy --target prod

# Run a job
databricks bundle run daily_revenue --target dev

# Destroy resources (cleanup)
databricks bundle destroy --target dev

# Show deployment status
databricks bundle summary --target prod
```

In [0]:
# DABs simulation
print("Declarative Automation Bundles (DABs) Quick Reference")
print("=" * 60)

dabs_workflow = {
    "1. Define": "Write databricks.yml with all resources",
    "2. Validate": "databricks bundle validate --target dev",
    "3. Deploy Dev": "databricks bundle deploy --target dev",
    "4. Test": "databricks bundle run smoke_tests --target dev",
    "5. Deploy Prod": "databricks bundle deploy --target prod",
    "6. Monitor": "Check job runs in Databricks UI",
}

print("\nDABs Workflow:")
for step, action in dabs_workflow.items():
    print(f"  {step}: {action}")

print("\nKey Benefits:")
benefits = [
    "Infrastructure as Code -- all resources in Git",
    "Environment parity -- same config, different targets",
    "CI/CD integration -- deploy via GitHub Actions",
    "Rollback -- git revert + bundle deploy",
    "Collaboration -- code review for infrastructure changes",
    "Audit trail -- Git history shows who changed what",
]
for b in benefits:
    print(f"  * {b}")

print("\nResource Types Supported:")
resource_types = [
    "jobs (Lakeflow Jobs)",
    "pipelines (Lakeflow Declarative Pipelines / DLT)",
    "clusters (cluster configurations)",
    "dashboards (Lakeview dashboards)",
    "experiments (MLflow experiments)",
    "model_serving_endpoints",
    "permissions (access control)",
]
for rt in resource_types:
    print(f"  * {rt}")


---
*Notebook 19 of the Databricks Data Engineering series*
*Study Order: 19*

---
# 📗 Section 3: DABs in CI/CD

## GitHub Actions with DABs

```yaml
# .github/workflows/deploy.yml
name: Deploy with DABs

on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Install Databricks CLI
        run: pip install databricks-cli
      
      - name: Validate bundle
        run: databricks bundle validate --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
      
      - name: Deploy to DEV
        run: databricks bundle deploy --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
      
      - name: Run smoke tests
        run: databricks bundle run smoke_tests --target dev
```

## Variable Substitution

```yaml
# databricks.yml
variables:
  env:
    default: dev
  cluster_size:
    default: "4"

resources:
  jobs:
    my_job:
      name: "[${var.env}] My Pipeline"
      tasks:
        - task_key: main
          new_cluster:
            num_workers: ${var.cluster_size}

# Override at deploy time:
# databricks bundle deploy --target prod --var="cluster_size=16"
```

In [0]:
# DABs workflow simulation
print("DABs Deployment Workflow")
print("=" * 60)

dabs_commands = {
    "Development": [
        "databricks bundle validate --target dev",
        "databricks bundle deploy --target dev",
        "databricks bundle run my_job --target dev",
    ],
    "Staging": [
        "databricks bundle validate --target staging",
        "databricks bundle deploy --target staging",
        "databricks bundle run integration_tests --target staging",
    ],
    "Production": [
        "# Requires manual approval in GitHub",
        "databricks bundle validate --target prod",
        "databricks bundle deploy --target prod",
        "databricks bundle run smoke_tests --target prod",
    ],
    "Rollback": [
        "git checkout v1.2.3  # Previous release tag",
        "databricks bundle deploy --target prod",
        "# Verify smoke tests pass",
    ],
}

for env, commands in dabs_commands.items():
    print(f"\n{env}:")
    for cmd in commands:
        print(f"  $ {cmd}")


---
# Section 3: Complete databricks.yml Reference

## Full Bundle Configuration

```yaml
bundle:
  name: techmart-pipeline

workspace:
  host: ${var.databricks_host}

variables:
  databricks_host:
    description: "Databricks workspace URL"
  env:
    description: "Deployment environment"
    default: dev
  cluster_workers:
    description: "Number of workers"
    default: "4"

resources:
  jobs:
    daily_revenue:
      name: "[${var.env}] Daily Revenue Pipeline"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "America/New_York"
        pause_status: UNPAUSED
      email_notifications:
        on_failure: ["data-team@company.com"]
      timeout_seconds: 7200
      tasks:
        - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/ingest_orders
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: ${var.cluster_workers}
          max_retries: 3
          min_retry_interval_millis: 300000
        - task_key: transform_silver
          depends_on:
            - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/transform_silver
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 8
        - task_key: build_gold
          depends_on:
            - task_key: transform_silver
          sql_task:
            query: "CALL gold.build_daily_revenue()"
            warehouse_id: "abc123"

  pipelines:
    medallion_pipeline:
      name: "[${var.env}] Medallion Pipeline"
      serverless: true
      catalog: prod
      target: medallion
      libraries:
        - notebook:
            path: ./src/bronze_layer
        - notebook:
            path: ./src/silver_layer

targets:
  dev:
    mode: development
    workspace:
      host: https://dev-workspace.cloud.databricks.com
  staging:
    workspace:
      host: https://staging-workspace.cloud.databricks.com
  prod:
    mode: production
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    resources:
      jobs:
        daily_revenue:
          tasks:
            - task_key: ingest_orders
              new_cluster:
                num_workers: 16
```

In [0]:
# DABs CLI commands reference
print("DABs CLI Commands Reference")
print("=" * 60)

commands = {
    "Validate": {
        "cmd": "databricks bundle validate --target dev",
        "purpose": "Check configuration for errors before deploying",
        "when": "Always run before deploy",
    },
    "Deploy to dev": {
        "cmd": "databricks bundle deploy --target dev",
        "purpose": "Deploy all resources to dev environment",
        "when": "After every code change",
    },
    "Deploy to prod": {
        "cmd": "databricks bundle deploy --target prod",
        "purpose": "Deploy to production",
        "when": "After staging validation passes",
    },
    "Run a job": {
        "cmd": "databricks bundle run daily_revenue --target dev",
        "purpose": "Trigger a job run",
        "when": "Testing or manual trigger",
    },
    "Run with params": {
        "cmd": 'databricks bundle run daily_revenue --target dev --python-params '["2024-03-15"]'',
        "purpose": "Run with custom parameters",
        "when": "Backfill or custom date",
    },
    "Override variable": {
        "cmd": "databricks bundle deploy --target prod --var=cluster_workers=16",
        "purpose": "Override a variable at deploy time",
        "when": "Environment-specific tuning",
    },
    "Destroy resources": {
        "cmd": "databricks bundle destroy --target dev",
        "purpose": "Remove all deployed resources",
        "when": "Cleanup dev environment",
    },
    "Show summary": {
        "cmd": "databricks bundle summary --target prod",
        "purpose": "Show what is deployed",
        "when": "Audit current state",
    },
}

for name, details in commands.items():
    print(f"\n{name}:")
    print(f"  $ {details['cmd']}")
    print(f"  Purpose: {details['purpose']}")
    print(f"  When: {details['when']}")


---
# Section 4: CI/CD with DABs and GitHub Actions

## Complete GitHub Actions Workflow

```yaml
# .github/workflows/deploy.yml
name: Deploy with DABs

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"
      - name: Install dependencies
        run: pip install -r requirements.txt pytest ruff
      - name: Lint
        run: ruff check src/
      - name: Unit tests
        run: pytest tests/unit/ -v
      - name: Validate bundle (dev)
        run: databricks bundle validate --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}

  deploy-dev:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to DEV
        run: databricks bundle deploy --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}

  deploy-prod:
    needs: deploy-dev
    runs-on: ubuntu-latest
    environment: production  # Requires manual approval
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to PROD
        run: databricks bundle deploy --target prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_PROD }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_PROD }}
```

In [0]:
# DABs deployment simulation
class DaBsDeployment:
    def __init__(self, bundle_name):
        self.bundle_name = bundle_name
        self.deployed = {}

    def validate(self, target):
        print(f"  Validating bundle for target: {target}")
        print(f"  Checking databricks.yml syntax... OK")
        print(f"  Checking notebook paths exist... OK")
        print(f"  Checking cluster configs... OK")
        return True

    def deploy(self, target, overrides=None):
        if not self.validate(target):
            return False
        print(f"  Deploying to {target}...")
        resources = ["daily_revenue (job)", "medallion_pipeline (pipeline)"]
        for r in resources:
            print(f"    Deployed: {r}")
        self.deployed[target] = resources
        return True

    def run(self, job_name, target):
        print(f"  Triggering: {job_name} on {target}")
        print(f"  Run ID: 12345")
        return "12345"


bundle = DaBsDeployment("techmart-pipeline")

print("DABs Deployment Workflow")
print("=" * 60)

print("\n--- PR opened: validate only ---")
bundle.validate("dev")

print("\n--- Merge to main: deploy to dev ---")
bundle.deploy("dev")

print("\n--- After dev tests pass: deploy to prod ---")
bundle.deploy("prod")

print("\n--- Manual trigger: run a job ---")
bundle.run("daily_revenue", "prod")


---
# Section 3: Complete databricks.yml Reference

## Full Bundle Configuration

```yaml
bundle:
  name: techmart-pipeline

workspace:
  host: ${var.databricks_host}

variables:
  databricks_host:
    description: "Databricks workspace URL"
  env:
    description: "Deployment environment"
    default: dev
  cluster_workers:
    description: "Number of workers"
    default: "4"

resources:
  jobs:
    daily_revenue:
      name: "[${var.env}] Daily Revenue Pipeline"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "America/New_York"
        pause_status: UNPAUSED
      email_notifications:
        on_failure: ["data-team@company.com"]
      timeout_seconds: 7200
      tasks:
        - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/ingest_orders
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: ${var.cluster_workers}
          max_retries: 3
          min_retry_interval_millis: 300000
        - task_key: transform_silver
          depends_on:
            - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/transform_silver
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 8
        - task_key: build_gold
          depends_on:
            - task_key: transform_silver
          sql_task:
            query: "CALL gold.build_daily_revenue()"
            warehouse_id: "abc123"

  pipelines:
    medallion_pipeline:
      name: "[${var.env}] Medallion Pipeline"
      serverless: true
      catalog: prod
      target: medallion
      libraries:
        - notebook:
            path: ./src/bronze_layer
        - notebook:
            path: ./src/silver_layer

targets:
  dev:
    mode: development
    workspace:
      host: https://dev-workspace.cloud.databricks.com
  staging:
    workspace:
      host: https://staging-workspace.cloud.databricks.com
  prod:
    mode: production
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    resources:
      jobs:
        daily_revenue:
          tasks:
            - task_key: ingest_orders
              new_cluster:
                num_workers: 16
```

In [0]:
# DABs CLI commands reference
print("DABs CLI Commands Reference")
print("=" * 60)

commands = {
    "Validate": {
        "cmd": "databricks bundle validate --target dev",
        "purpose": "Check configuration for errors before deploying",
        "when": "Always run before deploy",
    },
    "Deploy to dev": {
        "cmd": "databricks bundle deploy --target dev",
        "purpose": "Deploy all resources to dev environment",
        "when": "After every code change",
    },
    "Deploy to prod": {
        "cmd": "databricks bundle deploy --target prod",
        "purpose": "Deploy to production",
        "when": "After staging validation passes",
    },
    "Run a job": {
        "cmd": "databricks bundle run daily_revenue --target dev",
        "purpose": "Trigger a job run",
        "when": "Testing or manual trigger",
    },
    "Run with params": {
        "cmd": 'databricks bundle run daily_revenue --target dev --python-params '["2024-03-15"]'',
        "purpose": "Run with custom parameters",
        "when": "Backfill or custom date",
    },
    "Override variable": {
        "cmd": "databricks bundle deploy --target prod --var=cluster_workers=16",
        "purpose": "Override a variable at deploy time",
        "when": "Environment-specific tuning",
    },
    "Destroy resources": {
        "cmd": "databricks bundle destroy --target dev",
        "purpose": "Remove all deployed resources",
        "when": "Cleanup dev environment",
    },
    "Show summary": {
        "cmd": "databricks bundle summary --target prod",
        "purpose": "Show what is deployed",
        "when": "Audit current state",
    },
}

for name, details in commands.items():
    print(f"\n{name}:")
    print(f"  $ {details['cmd']}")
    print(f"  Purpose: {details['purpose']}")
    print(f"  When: {details['when']}")


---
# Section 4: CI/CD with DABs and GitHub Actions

## Complete GitHub Actions Workflow

```yaml
# .github/workflows/deploy.yml
name: Deploy with DABs

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"
      - name: Install dependencies
        run: pip install -r requirements.txt pytest ruff
      - name: Lint
        run: ruff check src/
      - name: Unit tests
        run: pytest tests/unit/ -v
      - name: Validate bundle (dev)
        run: databricks bundle validate --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}

  deploy-dev:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to DEV
        run: databricks bundle deploy --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}

  deploy-prod:
    needs: deploy-dev
    runs-on: ubuntu-latest
    environment: production  # Requires manual approval
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to PROD
        run: databricks bundle deploy --target prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_PROD }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_PROD }}
```

In [0]:
# DABs deployment simulation
class DaBsDeployment:
    def __init__(self, bundle_name):
        self.bundle_name = bundle_name
        self.deployed = {}

    def validate(self, target):
        print(f"  Validating bundle for target: {target}")
        print(f"  Checking databricks.yml syntax... OK")
        print(f"  Checking notebook paths exist... OK")
        print(f"  Checking cluster configs... OK")
        return True

    def deploy(self, target, overrides=None):
        if not self.validate(target):
            return False
        print(f"  Deploying to {target}...")
        resources = ["daily_revenue (job)", "medallion_pipeline (pipeline)"]
        for r in resources:
            print(f"    Deployed: {r}")
        self.deployed[target] = resources
        return True

    def run(self, job_name, target):
        print(f"  Triggering: {job_name} on {target}")
        print(f"  Run ID: 12345")
        return "12345"


bundle = DaBsDeployment("techmart-pipeline")

print("DABs Deployment Workflow")
print("=" * 60)

print("\n--- PR opened: validate only ---")
bundle.validate("dev")

print("\n--- Merge to main: deploy to dev ---")
bundle.deploy("dev")

print("\n--- After dev tests pass: deploy to prod ---")
bundle.deploy("prod")

print("\n--- Manual trigger: run a job ---")
bundle.run("daily_revenue", "prod")


---
# Section 3: Complete databricks.yml Reference

## Full Bundle Configuration

```yaml
bundle:
  name: techmart-pipeline

workspace:
  host: ${var.databricks_host}

variables:
  databricks_host:
    description: "Databricks workspace URL"
  env:
    description: "Deployment environment"
    default: dev
  cluster_workers:
    description: "Number of workers"
    default: "4"

resources:
  jobs:
    daily_revenue:
      name: "[${var.env}] Daily Revenue Pipeline"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "America/New_York"
        pause_status: UNPAUSED
      email_notifications:
        on_failure: ["data-team@company.com"]
      timeout_seconds: 7200
      tasks:
        - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/ingest_orders
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: ${var.cluster_workers}
          max_retries: 3
          min_retry_interval_millis: 300000
        - task_key: transform_silver
          depends_on:
            - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/transform_silver
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 8
        - task_key: build_gold
          depends_on:
            - task_key: transform_silver
          sql_task:
            query: "CALL gold.build_daily_revenue()"
            warehouse_id: "abc123"

  pipelines:
    medallion_pipeline:
      name: "[${var.env}] Medallion Pipeline"
      serverless: true
      catalog: prod
      target: medallion
      libraries:
        - notebook:
            path: ./src/bronze_layer
        - notebook:
            path: ./src/silver_layer

targets:
  dev:
    mode: development
    workspace:
      host: https://dev-workspace.cloud.databricks.com
  staging:
    workspace:
      host: https://staging-workspace.cloud.databricks.com
  prod:
    mode: production
    workspace:
      host: https://prod-workspace.cloud.databricks.com
    resources:
      jobs:
        daily_revenue:
          tasks:
            - task_key: ingest_orders
              new_cluster:
                num_workers: 16
```

In [0]:
# DABs CLI commands reference
print("DABs CLI Commands Reference")
print("=" * 60)

commands = {
    "Validate": {
        "cmd": "databricks bundle validate --target dev",
        "purpose": "Check configuration for errors before deploying",
        "when": "Always run before deploy",
    },
    "Deploy to dev": {
        "cmd": "databricks bundle deploy --target dev",
        "purpose": "Deploy all resources to dev environment",
        "when": "After every code change",
    },
    "Deploy to prod": {
        "cmd": "databricks bundle deploy --target prod",
        "purpose": "Deploy to production",
        "when": "After staging validation passes",
    },
    "Run a job": {
        "cmd": "databricks bundle run daily_revenue --target dev",
        "purpose": "Trigger a job run",
        "when": "Testing or manual trigger",
    },
    "Run with params": {
        "cmd": 'databricks bundle run daily_revenue --target dev --python-params '["2024-03-15"]'',
        "purpose": "Run with custom parameters",
        "when": "Backfill or custom date",
    },
    "Override variable": {
        "cmd": "databricks bundle deploy --target prod --var=cluster_workers=16",
        "purpose": "Override a variable at deploy time",
        "when": "Environment-specific tuning",
    },
    "Destroy resources": {
        "cmd": "databricks bundle destroy --target dev",
        "purpose": "Remove all deployed resources",
        "when": "Cleanup dev environment",
    },
    "Show summary": {
        "cmd": "databricks bundle summary --target prod",
        "purpose": "Show what is deployed",
        "when": "Audit current state",
    },
}

for name, details in commands.items():
    print(f"\n{name}:")
    print(f"  $ {details['cmd']}")
    print(f"  Purpose: {details['purpose']}")
    print(f"  When: {details['when']}")


---
# Section 4: CI/CD with DABs and GitHub Actions

## Complete GitHub Actions Workflow

```yaml
# .github/workflows/deploy.yml
name: Deploy with DABs

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"
      - name: Install dependencies
        run: pip install -r requirements.txt pytest ruff
      - name: Lint
        run: ruff check src/
      - name: Unit tests
        run: pytest tests/unit/ -v
      - name: Validate bundle (dev)
        run: databricks bundle validate --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}

  deploy-dev:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to DEV
        run: databricks bundle deploy --target dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_DEV }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_DEV }}

  deploy-prod:
    needs: deploy-dev
    runs-on: ubuntu-latest
    environment: production  # Requires manual approval
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to PROD
        run: databricks bundle deploy --target prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST_PROD }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN_PROD }}
```

In [0]:
# DABs deployment simulation
class DaBsDeployment:
    def __init__(self, bundle_name):
        self.bundle_name = bundle_name
        self.deployed = {}

    def validate(self, target):
        print(f"  Validating bundle for target: {target}")
        print(f"  Checking databricks.yml syntax... OK")
        print(f"  Checking notebook paths exist... OK")
        print(f"  Checking cluster configs... OK")
        return True

    def deploy(self, target, overrides=None):
        if not self.validate(target):
            return False
        print(f"  Deploying to {target}...")
        resources = ["daily_revenue (job)", "medallion_pipeline (pipeline)"]
        for r in resources:
            print(f"    Deployed: {r}")
        self.deployed[target] = resources
        return True

    def run(self, job_name, target):
        print(f"  Triggering: {job_name} on {target}")
        print(f"  Run ID: 12345")
        return "12345"


bundle = DaBsDeployment("techmart-pipeline")

print("DABs Deployment Workflow")
print("=" * 60)

print("\n--- PR opened: validate only ---")
bundle.validate("dev")

print("\n--- Merge to main: deploy to dev ---")
bundle.deploy("dev")

print("\n--- After dev tests pass: deploy to prod ---")
bundle.deploy("prod")

print("\n--- Manual trigger: run a job ---")
bundle.run("daily_revenue", "prod")


---
*Notebook 35 of the Databricks Data Engineering series*
*Study Order: 19*